# The Augmented LLM

## Load Keys in Env.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
MODEL_NAME = "command-a-03-2025"

In [5]:
import os
from cohere import ClientV2

co = ClientV2(api_key=os.environ["COHERE_API_KEY"])

## Describe And Define Tool

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluates a basic Python match expression and returns the numeric result",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A math expression, e.g. '23 * 57 + 5'",
                    }
                },
                "required": ["expression"],
            },
        },
    }
]


def calculator(expression: str) -> dict:
    return {
        "result": eval( # Never do eval in production
            expression,
            {
                "__builtins__": {},
            },
            {},
        )
    }


TOOL_REGISTRY = {
    "calculator": calculator,
}

## LLM Initial Call

In [6]:
messages = [
    {
        "role": "user",
        "content": "What is (1289 * 47)? Show your work briefly.",
    },
]

resp = co.chat(
    model=MODEL_NAME,
    messages=messages,
    tools=tools,
)

resp

V2ChatResponse(id='1a342758-1ac5-4579-81ef-7bce5e680acd', finish_reason='TOOL_CALL', message=AssistantMessageResponse(role='assistant', tool_calls=[ToolCallV2(id='calculator_r62v1yw1ch8x', type='function', function=ToolCallV2Function(name='calculator', arguments='{"expression":"1289 * 47"}'))], tool_plan='I will use the calculator tool to calculate the answer to the question.', content=None, citations=None), usage=Usage(billed_units=UsageBilledUnits(input_tokens=69.0, output_tokens=27.0, search_units=None, classifications=None), tokens=UsageTokens(input_tokens=1473.0, output_tokens=56.0), cached_tokens=112.0), logprobs=None)

## Tool Invocation And Context Update

In [ ]:
import json

if resp.message.tool_calls:
    print(f"Model wants to call a tool. Plan: {resp.message.tool_plan}")

    # Record the assistants tool-call turn so that model can see its own request the next time.
    messages.append(
        {
            "role": "assistant",
            "tool_calls": resp.message.tool_calls,
            "tool_plan": resp.message.tool_plan,
        }
    )

    # Execute each tool call and append the result.
    for tc in resp.message.tool_calls:
        args = json.loads(tc.function.arguments)
        result = TOOL_REGISTRY[tc.function.name](**args)
        print(f"->  {tc.function.name}({args}) = {result}")

        messages.append(
            {
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result),
            }
        )

    # Second call - now the model has the result and writes the final answer.
    resp = co.chat(
        model=MODEL_NAME,
        messages=messages,
        tools=tools,
    )

print(f"\nFinal answer: {resp.message.content[0].text}\n")

Model wants to call a tool. Plan: I will use the calculator tool to calculate the answer to the question.
->  calculator({'expression': '1289 * 47'}) = {'result': 60583}

Final answer: 1289 * 47 = **60583**.

